In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, unix_timestamp, lit
from pyspark.ml.feature import StringIndexer

# Initialize Spark session
spark = SparkSession.builder \
    .appName("LendingClubPreprocessing") \
    .getOrCreate()

25/11/09 23:22:41 WARN Utils: Your hostname, Parikshits-MacBook-Pro-6.local resolves to a loopback address: 127.0.0.1; using 10.0.0.55 instead (on interface en0)
25/11/09 23:22:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/11/09 23:22:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
def clean_string_columns(df):
    if 'int_rate' in df.columns:
        df = df.withColumn('int_rate', (regexp_extract('int_rate', r'(\d+\.?\d*)', 0).cast('float') / 100))

    if 'revol_util' in df.columns:
        df = df.withColumn('revol_util', regexp_extract('revol_util', r'(\d+\.?\d*)', 0).cast('float'))

    if 'emp_length' in df.columns:
        df = df.withColumn('emp_length', regexp_extract('emp_length', r'(\d+)', 0).cast('int'))

    if 'term' in df.columns:
        df = df.withColumn('term', regexp_extract('term', r'(\d+)', 0).cast('int'))

    return df

In [3]:
def unify_fico_score(df):
    fico_cols = [col for col in df.columns if 'fico' in col.lower()]

    if fico_cols:
        if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
            df = df.withColumn('fico_score', (col('fico_range_low') + col('fico_range_high')) / 2)
        elif len(fico_cols) > 0:
            df = df.withColumn('fico_score', col(fico_cols[0]))
    else:
        df = df.withColumn('fico_score', lit(700))  # Default FICO score

    return df

In [4]:
def handle_missing_values(df):
    
    numeric_columns = [col for col, dtype in df.dtypes if dtype in ('int', 'double')]
    for col_name in numeric_columns:
        non_null_count = df.filter(df[col_name].isNotNull()).count()
        if non_null_count > 0:
            median_value = df.approxQuantile(col_name, [0.5], 0)[0]
            df = df.fillna({col_name: median_value})
        else:
            print(f"Column {col_name} has no non-null values, skipping median fill.")
    
    categorical_columns = [col for col, dtype in df.dtypes if dtype == 'string']
    for col_name in categorical_columns:
        mode_value_row = df.groupby(col_name).count().orderBy("count", ascending=False).first()
        
        if mode_value_row is not None:
            mode_value = mode_value_row[0]
            if mode_value is not None:
                df = df.fillna({col_name: mode_value})
            else:
                df = df.fillna({col_name: ""})
        else:
            df = df.fillna({col_name: ""})

    return df

In [5]:
def encode_categorical_columns(df, categorical_columns):
    encoded_columns = []
    for cat_feature in categorical_columns:
        encoded_column_name = f'{cat_feature}_encoded'
        if encoded_column_name in df.columns:
            df = df.drop(encoded_column_name)
        
        encoder = StringIndexer(inputCol=cat_feature, outputCol=encoded_column_name)
        df = encoder.fit(df).transform(df)
        encoded_columns.append(encoded_column_name)
    
    return df, encoded_columns

In [6]:
def feature_engineering(df):
    df = df.withColumn('income_to_loan_ratio', col('annual_inc') / col('loan_amnt'))
    
    if 'issue_d' in df.columns:
        df = df.withColumn('issue_d', unix_timestamp('issue_d', 'yyyy-MM-dd').cast('timestamp'))
        df = df.withColumn('days_since_issue', (unix_timestamp() - unix_timestamp('issue_d')) / 86400)  # In days

    return df

In [7]:
def finalize_features(df, categorical_columns, encoded_columns):
    """Finalizes the set of features that will be used for training."""
    numeric_columns = [col for col in df.columns if col not in categorical_columns]
    final_features = numeric_columns + encoded_columns
    return df.select(final_features)

In [8]:
import os
from pyspark.sql.utils import AnalysisException

def remove_duplicate_columns(df):
    cols = df.columns
    seen = set()
    duplicates = []
    for col in cols:
        if col in seen:
            duplicates.append(col)
        else:
            seen.add(col)
    if duplicates:
        print(f"Duplicate columns found: {duplicates}")
        for col in duplicates:
            df = df.drop(col)
    return df

def preprocess_by_year(df, categorical_columns, target_column):
    for cat_feature in categorical_columns:
        encoded_column_name = f'{cat_feature}_encoded'
        if encoded_column_name in df.columns:
            print(f"Dropping {encoded_column_name} column.") 
            df = df.drop(encoded_column_name)

    df = clean_string_columns(df)   
    df = unify_fico_score(df)       
    df = handle_missing_values(df)  
    df = feature_engineering(df)    

    df, encoded_columns = encode_categorical_columns(df, categorical_columns)

    df = finalize_features(df, categorical_columns, encoded_columns)

    df = remove_duplicate_columns(df)

    return df

categorical_columns = ['home_ownership', 'purpose']
target_column = 'loan_status_binary'

df = spark.read.parquet("../../accepted_by_year") 

years = df.select("year").distinct().rdd.flatMap(lambda x: x).collect()

for year in years:
    df_year = df.filter(df["year"] == year)
    
    print(f"Columns in raw data for year {year}: {df_year.columns}")

    df_year_preprocessed = preprocess_by_year(df_year, categorical_columns, target_column)
    
    print(f"Columns before saving for year {year}: {df_year_preprocessed.columns}")

    output_path = f'../../preprocessed/year={year}'
    if not os.path.exists(output_path):
        os.makedirs(output_path) 
    
    try:
        df_year_preprocessed.write.mode("overwrite").parquet(output_path)
        print(f"Preprocessed data for year {year} saved.")
    except AnalysisException as e:
        print(f"Error occurred while saving data for year {year}: {e}")


Columns in raw data for year 2018: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verificati

25/11/09 23:22:57 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
Duplicate columns found: ['home_ownership_encoded', 'purpose_encoded']
Columns before saving for year 2018: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec

Preprocessed data for year 2018 saved.
Columns in raw data for year 2015: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2015 saved.
Columns in raw data for year 2016: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2016 saved.
Columns in raw data for year 2017: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2017 saved.
Columns in raw data for year 2014: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2014 saved.
Columns in raw data for year 2013: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2013 saved.
Columns in raw data for year 2012: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2012 saved.
Columns in raw data for year 2007: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2010 saved.
Columns in raw data for year 2011: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu

Preprocessed data for year 2011 saved.
Columns in raw data for year 2008: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annu